# ADS v2: Travel acquisition / renewal events

Builds 7 event families: `TA1`, `TA1p`, `TA5`, `TA3.B1.2`, `TR1.E1`, `TR4`, `TR.O1`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("18_Travel", "started")


## TA1: Travel quote initiated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ta1  (TA1)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 397. Travel quotes created in the window, certi */
/*              : ficate status Completed Quote, Incomplete Quote, Cancelled or Live, with an email; at */
/*              :  the quote date. Change: one event per quote; 04 grouped by quote_number, so every in */
/*              : complete quote (quote_number 0) collapsed into one event. ods_v2 keys those by hashed */
/*              :  email + travel date. EmailKey is null for badly formed emails, slightly stricter tha */
/*              : n 04's len(email) > 0.                                                                */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Travel   Source: QuoteTravel                                  */
/*              : Reads: ods_v2.QuoteTravel                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ta1 as
select  xxhash64('TA1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TA1'                                               as EventTypeId,
        'Travel quote initiated' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  q.QuoteDate                                                     as EventDateTime,
                8                                                               as SourceSystemId,
                q.SourceSystemQuoteReference                                    as SourceQuoteReference,
                q.EmailKey                                                      as SourceCustomerReference,
                'EmailKey'                                                      as SourceCustomerReferenceType,
                q.PolicyId,
                q.QuoteId
        from    ${ads.ods}.QuoteTravel q
        where   trim(q.CertificateStatus) in ('Completed Quote', 'Incomplete Quote', 'Cancelled', 'Live')
        and     q.EmailKey is not null
        and     q.QuoteDate >= timestamp '${ads.start_time}' and q.QuoteDate < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TA1p: Travel Acquisition - Travel Quote Completed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ta1p  (TA1p)                                                            */
/*  Description : Acquisition / Spine. Notebook 04 cells 59 (quote leg of stg.TravelQuotes), 398. New b */
/*              : usiness travel quotes created in the window with status Completed Quote, Incomplete Q */
/*              : uote or Cancelled (04's quote leg left out Live), an email and a purchase date; at th */
/*              : e purchase date. Change: 04's stg.TravelQuotes also unioned two policy legs, which ma */
/*              : de TA1p a copy of TA5 (T2, P2); the purchase from the policy file is TA5 (D-E1.23), s */
/*              : o TA1p keeps the quote file's leg only. Decision needed: whether "quote completed" sh */
/*              : ould rather be status Completed Quote at its quote time.                              */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Travel   Source: QuoteTravel                                  */
/*              : Reads: ods_v2.QuoteTravel                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ta1p as
select  xxhash64('TA1p', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TA1p'                                              as EventTypeId,
        'Travel Acquisition - Travel Quote Completed' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  q.PurchaseDate                                                  as EventDateTime,
                8                                                               as SourceSystemId,
                q.SourceSystemQuoteReference                                    as SourceQuoteReference,
                q.EmailKey                                                      as SourceCustomerReference,
                'EmailKey'                                                      as SourceCustomerReferenceType,
                q.PolicyId,
                q.QuoteId
        from    ${ads.ods}.QuoteTravel q
        where   trim(q.CertificateStatus) in ('Completed Quote', 'Incomplete Quote', 'Cancelled')
        and     coalesce(q.BusinessType, '') <> 'Renewal'
        and     q.EmailKey is not null
        and     q.PurchaseDate is not null
        and     q.QuoteDate >= timestamp '${ads.start_time}' and q.QuoteDate < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TA5: Purchase confirmed by next-day CSV


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ta5  (TA5)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 59, 399. New business purchases confirmed by t */
/*              : he Mapfre policy file: transactions with record type N (new policy) and RenewalFlag 0 */
/*              : , purchased in the window; one event per Mapfre policy at its purchase date (D-E1.23, */
/*              :  T2). Change: 04 read stg.TravelQuotes (quotes plus two policy legs), so a purchase c */
/*              : ould count twice (483,191 TA5 against 351,688 TA1); U (update) and C (void) records a */
/*              : re not purchases.                                                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Travel   Source: PolicyTravelTransaction                     */
/*              : Reads: ods_v2.PolicyTravelTransaction                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ta5 as
select  xxhash64('TA5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TA5'                                               as EventTypeId,
        'Purchase confirmed by next-day CSV' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(t.PurchaseDate)                                             as EventDateTime,
                5                                                               as SourceSystemId,
                cast(t.MapfrePolicyId as string)                                as SourcePolicyReference,
                cast(min(t.MapfreQuoteId) as string)                            as SourceQuoteReference,
                cast(min(t.MapfreCustomerId) as string)                         as SourceCustomerReference,
                'MapfreCustomerId'                                              as SourceCustomerReferenceType,
                min(t.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyTravelTransaction t
        where   t.RecordType = 'N'
        and     coalesce(t.RenewalFlag, 0) = 0
        and     t.PurchaseDate >= timestamp '${ads.start_time}' and t.PurchaseDate < timestamp '${ads.end_time}'
        group by t.MapfrePolicyId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TA3.B1.2: Purchase


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ta3_b1_2  (TA3.B1.2)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 400. New business purchases as the warehouse's */
/*              :  travel quotes-and-sales fact holds them (quotes with business type New Business and  */
/*              : a purchase date in the window); at the purchase date. Change: read from QuoteTravel ( */
/*              : the fact's quote side), not edw.tbl_fact_TravelAllQuotesSales at the effective-date s */
/*              : napshot (edw is not a source for ads). Decision needed (T2): with the policy file as  */
/*              : the purchase source of record (TA5), this step repeats TA5 from the quote side; keep  */
/*              : it, or derive it from TA5.                                                            */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Travel   Source: QuoteTravel                                  */
/*              : Reads: ods_v2.QuoteTravel                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ta3_b1_2 as
select  xxhash64('TA3.B1.2', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TA3.B1.2'                                          as EventTypeId,
        'Purchase' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  q.PurchaseDate                                                  as EventDateTime,
                8                                                               as SourceSystemId,
                q.SourceSystemQuoteReference                                    as SourceQuoteReference,
                q.EmailKey                                                      as SourceCustomerReference,
                case when q.EmailKey is not null then 'EmailKey' end            as SourceCustomerReferenceType,
                q.PolicyId,
                q.QuoteId
        from    ${ads.ods}.QuoteTravel q
        where   q.BusinessType = 'New Business'
        and     q.PurchaseDate >= timestamp '${ads.start_time}' and q.PurchaseDate < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TR1.E1: Travel renewal date campaign


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_tr1_e1  (TR1.E1)                                                        */
/*  Description : Renewals / Chill Effort. Notebook 04 cell 401. Travel renewal-date campaigns ('Travel */
/*              :  - Renewal Date in the next 14 Days' / '... next 7 Days', two spaces as the campaign  */
/*              : is named) sent by email or SMS; one event per policy, campaign and send day, at the f */
/*              : irst send (D-E1.20, T4: keyed to the policy). Change: 04 kept one event per raw email */
/*              :  address, its first ever message of any interaction type; here the customer is the Ma */
/*              : pfre customer id from the message's client code ('M' + id), else its EmailKey, and op */
/*              : ens / clicks / bounces are not sends.                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Travel   Source: MessageEvent                                */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_tr1_e1 as
select  xxhash64('TR1.E1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TR1.E1'                                            as EventTypeId,
        'Travel renewal date campaign' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                coalesce(case when m.ClientCode like 'M%' then substr(m.ClientCode, 2) end, m.EmailKey)
                                                                                as SourceCustomerReference,
                case when m.ClientCode like 'M%' then 'MapfreCustomerId'
                     when m.EmailKey is not null then 'EmailKey' end            as SourceCustomerReferenceType
        from    ${ads.ods}.MessageEvent m
        where   m.CampaignName in ('Travel - Renewal Date in the next 14 Days', 'Travel - Renewal Date in the next  7 Days')
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= timestamp '${ads.start_time}' and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode,
                coalesce(case when m.ClientCode like 'M%' then substr(m.ClientCode, 2) end, m.EmailKey),
                case when m.ClientCode like 'M%' then 'MapfreCustomerId' when m.EmailKey is not null then 'EmailKey' end,
                m.CampaignName, cast(m.EventDateTime as date)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TR4: Renewal processed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_tr4  (TR4)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cells 59, 402. Renewals processed: Mapfre policy transa */
/*              : ctions with record type N and RenewalFlag 1, purchased in the window; one event per r */
/*              : enewed policy at its purchase date (D-E1.23, T2, T4). A travel renewal is a re-purcha */
/*              : se under a new Mapfre policy id. Change: 04 read stg.TravelQuotes (renewal quotes plu */
/*              : s policy legs).                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Travel   Source: PolicyTravelTransaction                     */
/*              : Reads: ods_v2.PolicyTravelTransaction                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_tr4 as
select  xxhash64('TR4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TR4'                                               as EventTypeId,
        'Renewal processed' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(t.PurchaseDate)                                             as EventDateTime,
                5                                                               as SourceSystemId,
                cast(t.MapfrePolicyId as string)                                as SourcePolicyReference,
                cast(min(t.MapfreQuoteId) as string)                            as SourceQuoteReference,
                cast(min(t.MapfreCustomerId) as string)                         as SourceCustomerReference,
                'MapfreCustomerId'                                              as SourceCustomerReferenceType,
                min(t.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyTravelTransaction t
        where   t.RecordType = 'N'
        and     t.RenewalFlag = 1
        and     t.PurchaseDate >= timestamp '${ads.start_time}' and t.PurchaseDate < timestamp '${ads.end_time}'
        group by t.MapfrePolicyId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## TR.O1: Policy continues-  re-purchased


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_tr_o1  (TR.O1)                                                          */
/*  Description : Renewals / Outcome. Notebook 04 cell 403. Policy continues, re-purchased: Mapfre poli */
/*              : cies (latest record) with RenewalFlag 1 sold in the window, one event per Mapfre quot */
/*              : e at the purchase date, as 04 (dlk.EXT_Travel_Policy). Change: one row per event (04  */
/*              : wrote it twice, 50% exact repeats). Decision needed (T4): this is the same population */
/*              :  as TR4 (a travel renewal is a re-purchase); keep both, or make TR.O1 the per-cycle o */
/*              : utcome of TR4.                                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Travel   Source: PolicyTravel                                */
/*              : Reads: ods_v2.PolicyTravel                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_tr_o1 as
select  xxhash64('TR.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'TR.O1'                                             as EventTypeId,
        'Policy continues-  re-purchased' as EventDescription,
        'Travel'                                            as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(p.SalesDate)                                                as EventDateTime,
                5                                                               as SourceSystemId,
                cast(min(p.MapfrePolicyId) as string)                           as SourcePolicyReference,
                cast(p.MapfreQuoteId as string)                                 as SourceQuoteReference,
                cast(min(p.MapfreCustomerId) as string)                         as SourceCustomerReference,
                'MapfreCustomerId'                                              as SourceCustomerReferenceType,
                min(p.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyTravel p
        where   p.RenewalFlag = 1
        and     p.SalesDate >= timestamp '${ads.start_time}' and p.SalesDate < timestamp '${ads.end_time}'
        group by p.MapfreQuoteId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_ta1", "adsv2_evt_ta1p", "adsv2_evt_ta5", "adsv2_evt_ta3_b1_2", "adsv2_evt_tr1_e1", "adsv2_evt_tr4", "adsv2_evt_tr_o1"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("18_Travel", "succeeded")
